# WB → сцепка артикулов → новые цены Ozon

Токен вводится вручную. Шаблон Ozon предварительно скачивается из ЛК. Ноутбук создаёт копию XLSX и CSV с расчётом; цены в кабинете Ozon не меняет. Цель — WB с Кошельком −3%, корректировка в обе стороны.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass
import json
import sys

repo_root = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / "price_equalizer.py").is_file()), None)
if repo_root is None:
    raise RuntimeError("Откройте ноутбук из папки great_equalizer или great_equalizer/notebooks")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from wb_prices import BrowserContext
from price_equalizer import articles_for_template, fetch_template_snapshot, plan_equalization, write_outputs


## Входные файлы

Поместите таблицу сцепки и свежий шаблон Ozon в папку `inputs` или укажите собственные пути. Назначение колонок проверяется по заголовкам.


In [ ]:
mapping_path = repo_root / "inputs" / "mapping.xlsx"
template_path = repo_root / "inputs" / "ozon_template.xlsx"
dest = "-1257786"
target_mode = "wallet"
direction = "both"

articles = articles_for_template(mapping_path, template_path)
print("Артикулов WB для запроса:", len(articles))


## Данные сессии WB

Используйте рабочий `x_wbaas_token`, `deviceid` и User-Agent из своего запроса. Значения не вставляются в исходный код ячеек и не записываются в выходные файлы.


In [ ]:
token = getpass("x_wbaas_token: ")
deviceid = getpass("deviceid: ")
user_agent = input("User-Agent: ").strip()
context = BrowserContext.create(token, deviceid, user_agent)
del token, deviceid


## Получение цен и расчёт

Запросы выполняются группами по 50 артикулов с паузой между группами. Это выбранный размер запроса, а не заявленный лимит WB. При ошибке HTTP выполнение останавливается. Для неоднозначных размеров и отсутствующих связей AY останется пустой.


In [ ]:
snapshot = fetch_template_snapshot(mapping_path, template_path, context, dest=dest)
plan = plan_equalization(mapping_path, template_path, snapshot, target_mode=target_mode, direction=direction)
print("Статусы:", plan["summary"])
print("Изменений AY:", len(plan["updates"]))


In [ ]:
output_dir = repo_root / "outputs" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output_dir.mkdir(parents=True, exist_ok=False)
output_path = output_dir / "ozon_updated.xlsx"
audit_path = output_dir / "calculation.csv"
write_outputs(template_path, plan, output_path, audit_path)
with (output_dir / "wb_snapshot.json").open("x", encoding="utf-8") as handle:
    json.dump(snapshot, handle, ensure_ascii=False, indent=2)
print("Файл для Ozon:", output_path)
print("Расчёт и причины пропусков:", audit_path)


In [ ]:
[{key: row.get(key) for key in ("ozon_sku", "wb_id", "realization_price", "target_wb_rub", "new_limit_price_rub", "modeled_realization_rub", "status")} for row in plan["audit"][:15]]


Формула: `AY = цель WB × P / R`. Отношение `R / P` — коэффициент оставшейся цены. Модель предполагает, что этот коэффициент сохранится. После ручной загрузки файла в Ozon скачайте свежий шаблон и сравните фактическую цену реализации с целью WB: скидки и акции могут измениться.
